In [1]:
!pip install streamlit -q
!npm install -g localtunnel -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 44.0 MB/s eta 0:00:00
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸
added 22 packages in 3s
⠸
⠸3 packages are looking for funding
⠸  run `npm fund` for details
⠸

In [5]:
%%writefile app.py

import streamlit as st
import joblib
import pandas as pd
import numpy as np

# ── Load model & scaler ──────────────────────────────────────────────
model  = joblib.load('best_model.joblib')
scaler = joblib.load('scaler.joblib')

FEATURE_COLS = [
    'age_years', 'gender', 'height', 'weight',
    'ap_hi', 'ap_lo', 'cholesterol', 'gluc',
    'smoke', 'alco', 'active',
    'BMI', 'pulse_pressure', 'MAP', 'BMI_category'
]

MODEL_ACCURACY = "73.00%"   # ← ganti dengan accuracy model kamu setelah training
MODEL_F1       = "0.7300"   # ← ganti dengan F1-Score model kamu setelah training

# ── Halaman config ───────────────────────────────────────────────────
st.set_page_config(
    page_title="Cardiovascular Disease Prediction",
    page_icon="❤️",
    layout="centered"
)

# ── Header ───────────────────────────────────────────────────────────
st.title("❤️ Cardiovascular Disease Prediction")
st.markdown(
    "Masukkan data kesehatan pasien untuk memprediksi risiko "
    "**penyakit kardiovaskular**."
)
st.divider()

# ── Info performa model ───────────────────────────────────────────────
col_a, col_b = st.columns(2)
col_a.metric("Model Accuracy", MODEL_ACCURACY)
col_b.metric("F1-Score", MODEL_F1)
st.caption("Model: Random Forest Classifier | Dataset: Cardiovascular Disease (Kaggle, 70.000 data)")
st.divider()

# ── Input Form ───────────────────────────────────────────────────────
st.subheader("📋 Data Pasien")

col1, col2 = st.columns(2)

with col1:
    age      = st.number_input("Umur (tahun)", min_value=1, max_value=120, value=45)
    gender   = st.selectbox("Jenis Kelamin", options=["Perempuan", "Laki-laki"])
    height   = st.number_input("Tinggi Badan (cm)", min_value=100, max_value=250, value=165)
    weight   = st.number_input("Berat Badan (kg)", min_value=10.0, max_value=200.0, value=70.0, step=0.5)
    ap_hi    = st.number_input("Tekanan Darah Sistolik (ap_hi)", min_value=60, max_value=250, value=120)
    ap_lo    = st.number_input("Tekanan Darah Diastolik (ap_lo)", min_value=40, max_value=200, value=80)
    cholesterol = st.selectbox("Kolesterol", options=["Normal (1)", "Di atas normal (2)", "Jauh di atas normal (3)"])
    gluc     = st.selectbox("Glukosa", options=["Normal (1)", "Di atas normal (2)", "Jauh di atas normal (3)"])

with col2:
    smoke    = st.selectbox("Merokok?", options=["Tidak", "Ya"])
    alco     = st.selectbox("Konsumsi Alkohol?", options=["Tidak", "Ya"])
    active   = st.selectbox("Aktif Berolahraga?", options=["Tidak", "Ya"])

    # Hitung fitur turunan (preview)
    bmi_val = weight / ((height / 100) ** 2)
    pp_val  = ap_hi - ap_lo
    map_val = (ap_hi + 2 * ap_lo) / 3
    if bmi_val < 18.5:   bmi_cat = 0
    elif bmi_val < 25:   bmi_cat = 1
    elif bmi_val < 30:   bmi_cat = 2
    else:                bmi_cat = 3
    bmi_labels = {0: "Underweight", 1: "Normal", 2: "Overweight", 3: "Obese"}

    st.markdown("**📊 Nilai Turunan (otomatis dihitung):**")
    st.info(
        f"BMI: **{bmi_val:.1f}** ({bmi_labels[bmi_cat]})  \n"
        f"Pulse Pressure: **{pp_val}**  \n"
        f"MAP: **{map_val:.1f}**"
    )

st.divider()

# ── Prediksi ─────────────────────────────────────────────────────────
if st.button("🔍 Prediksi Sekarang", use_container_width=True, type="primary"):

    # Konversi input
    gender_val = 1 if gender == "Laki-laki" else 2
    chol_val   = int(cholesterol[0])
    gluc_val   = int(gluc[0])
    smoke_val  = 1 if smoke == "Ya" else 0
    alco_val   = 1 if alco  == "Ya" else 0
    active_val = 1 if active == "Ya" else 0

    input_data = pd.DataFrame([[
        float(age), gender_val, float(height), float(weight),
        float(ap_hi), float(ap_lo), chol_val, gluc_val,
        smoke_val, alco_val, active_val,
        bmi_val, float(pp_val), map_val, bmi_cat
    ]], columns=FEATURE_COLS)

    input_scaled = scaler.transform(input_data)
    prediction   = model.predict(input_scaled)[0]
    proba        = model.predict_proba(input_scaled)[0]

    st.subheader("📊 Hasil Prediksi")
    if prediction == 1:
        st.error(
            f"⚠️ **BERISIKO** terkena penyakit kardiovaskular\n\n"
            f"Probabilitas risiko: **{proba[1]*100:.1f}%**"
        )
        st.markdown(
            "**Rekomendasi:** Segera konsultasikan dengan dokter, "
            "jaga pola makan, dan rutin berolahraga."
        )
    else:
        st.success(
            f"✅ **TIDAK BERISIKO** terkena penyakit kardiovaskular\n\n"
            f"Probabilitas aman: **{proba[0]*100:.1f}%**"
        )
        st.markdown(
            "**Rekomendasi:** Tetap jaga gaya hidup sehat dan "
            "lakukan pemeriksaan rutin."
        )

    # Detail probabilitas
    st.markdown("**Detail probabilitas:**")
    prob_df = pd.DataFrame({
        'Kelas': ['Tidak Berisiko (0)', 'Berisiko (1)'],
        'Probabilitas': [f"{proba[0]*100:.1f}%", f"{proba[1]*100:.1f}%"]
    })
    st.table(prob_df)

# ── Footer ───────────────────────────────────────────────────────────
st.divider()
st.caption(
    "⚠️ Disclaimer: Prediksi ini hanya untuk keperluan akademik "
    "dan tidak menggantikan diagnosis medis profesional."
)

Overwriting app.py


In [7]:
!streamlit run app.py &>/dev/null &

In [ ]:
# Mendapatkan IP publik Colab Anda (diperlukan sebagai password LocalTunnel)
!wget -qO- ipv4.icanhazip.com

# Menjalankan LocalTunnel di port 8501
!npx localtunnel --port 8501

35.196.238.121
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹your url is: https://violet-mirrors-pull.loca.lt
